In [1]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
load_dotenv('../.env')
engine = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
print("Connected to database")

Connected to database


In [2]:
## Load Race Data
#Added `race_id` (proper group key for ranking) and kept `driver_id`, `finish_position`, `status` through to the end — all needed for v2, dropped in v1's model_df.

In [3]:
query = """
SELECT 
    rr.race_id,
    rr.driver_id,
    rr.constructor_id,
    rr.grid_position,
    rr.finish_position,
    rr.points,
    rr.points_finish,
    rr.status,
    r.season,
    r.round,
    r.circuit_id,
    r.date,
    q.q1_time,
    q.q2_time,
    q.q3_time,
    w.air_temp,
    w.track_temp,
    w.rainfall,
    w.humidity
FROM race_results rr
JOIN races r ON rr.race_id = r.race_id
LEFT JOIN qualifying_results q ON q.race_id = rr.race_id AND q.driver_id = rr.driver_id
LEFT JOIN weather w ON w.race_id = rr.race_id
ORDER BY r.date, rr.finish_position
"""
df = pd.read_sql(query, engine)
print(df.shape)
df.head()

(1838, 19)


,race_id,driver_id,constructor_id,grid_position,finish_position,points,points_finish,status,season,round,circuit_id,date,q1_time,q2_time,q3_time,air_temp,track_temp,rainfall,humidity
0,72,leclerc,ferrari,1.0,1.0,26.0,True,Finished,2022,1,Sakhir,2022-03-20,91.471,90.932,90.558,23.617791,28.610429,False,29.490798
1,72,sainz,ferrari,3.0,2.0,18.0,True,Finished,2022,1,Sakhir,2022-03-20,91.567,90.787,90.687,23.617791,28.610429,False,29.490798
2,72,hamilton,mercedes,5.0,3.0,15.0,True,Finished,2022,1,Sakhir,2022-03-20,92.285,91.048,91.238,23.617791,28.610429,False,29.490798
3,72,russell,mercedes,9.0,4.0,12.0,True,Finished,2022,1,Sakhir,2022-03-20,92.269,91.252,92.216,23.617791,28.610429,False,29.490798
4,72,kevin_magnussen,haas,7.0,5.0,10.0,True,Finished,2022,1,Sakhir,2022-03-20,91.955,91.461,91.808,23.617791,28.610429,False,29.490798


In [4]:
## Sanity Check: No Duplicate Driver-Race Rows

In [5]:
pd.read_sql("""
    SELECT driver_id, race_id, COUNT(*) as count 
    FROM race_results 
    GROUP BY driver_id, race_id 
    HAVING COUNT(*) > 1
    LIMIT 5
""", engine)

,driver_id,race_id,count


In [6]:
## Check Nulls Before Feature Engineering

In [7]:
df.isnull().sum()

race_id              0
driver_id            0
constructor_id       0
grid_position        2
finish_position      2
points               0
points_finish        0
status               0
season               0
round                0
circuit_id           0
date                 0
q1_time             21
q2_time            476
q3_time            941
air_temp             0
track_temp           0
rainfall             0
humidity             0
dtype: int64

In [8]:
## Qualifying Progression Flags

In [9]:
df['made_q2'] = df['q2_time'].notna().astype(int)
df['made_q3'] = df['q3_time'].notna().astype(int)

df[['q2_time', 'made_q2', 'q3_time', 'made_q3']].head(15)

,q2_time,made_q2,q3_time,made_q3
0,90.932,1,90.558,1
1,90.787,1,90.687,1
2,91.048,1,91.238,1
3,91.252,1,92.216,1
4,91.461,1,91.808,1
5,91.717,1,91.560,1
6,91.782,1,NaN,0
7,NaN,0,NaN,0
8,91.621,1,92.195,1
9,93.543,1,NaN,0


In [10]:
## Qualifying Gap to Pole

In [11]:
pole_times = df[df['grid_position'] == 1]
pole_times = pole_times[['season', 'round', 'q3_time']]
pole_times = pole_times.rename(columns={'q3_time': 'pole_time'})
df = df.merge(pole_times, on=['season', 'round'], how='left')
df['quali_gap_to_pole'] = df['q3_time'] - df['pole_time']

df['quali_gap_to_pole'] = df.groupby(['season', 'round'])['quali_gap_to_pole'].transform(
    lambda x: x.fillna(x.max() + 1.0)
)

In [12]:
## Driver Rolling Average Finish (last 5 races, shifted)
#Sort key: `['driver_id', 'date']` — required for shift() to grab a genuinely earlier race, not another driver's same-race result.

In [13]:
df = df.sort_values(['driver_id', 'date']).reset_index(drop=True)
df['driver_rolling_avg_finish'] = (
    df.groupby('driver_id')['finish_position']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

df[['driver_id', 'date', 'finish_position', 'driver_rolling_avg_finish']].head(20)

,driver_id,date,finish_position,driver_rolling_avg_finish
0,albon,2022-03-20,13.0,NaN
1,albon,2022-03-27,14.0,13.000000
2,albon,2022-04-10,10.0,13.500000
3,albon,2022-04-24,11.0,12.333333
4,albon,2022-05-08,9.0,12.000000
5,albon,2022-05-22,18.0,11.400000
6,albon,2022-05-29,18.0,12.400000
7,albon,2022-06-12,12.0,13.200000
8,albon,2022-06-19,13.0,13.600000
9,albon,2022-07-03,20.0,14.000000


In [14]:
## Constructor Rolling Average Finish (last 5 races, shifted)
#Sort key changes to `['constructor_id', 'date', 'driver_id']` — note this re-sort changes row order, which matters later for the group array.

In [15]:
df = df.sort_values(['constructor_id', 'date', 'driver_id']).reset_index(drop=True)
df['constructor_rolling_avg_finish'] = (
    df.groupby('constructor_id')['finish_position']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
df[df['constructor_id'] == 'williams'][['driver_id', 'date', 'finish_position', 'constructor_rolling_avg_finish']].head(20)

,driver_id,date,finish_position,constructor_rolling_avg_finish
1655,albon,2022-03-20,13.0,NaN
1656,latifi,2022-03-20,16.0,13.000000
1657,albon,2022-03-27,14.0,14.500000
1658,latifi,2022-03-27,18.0,14.333333
1659,albon,2022-04-10,10.0,15.250000
1660,latifi,2022-04-10,16.0,14.200000
1661,albon,2022-04-24,11.0,14.800000
1662,latifi,2022-04-24,16.0,13.800000
1663,albon,2022-05-08,9.0,14.200000
1664,latifi,2022-05-08,14.0,12.400000


In [16]:
## Driver Career Average & Circuit-Specific Average
#Sort key back to `['driver_id', 'date']`.

In [17]:
df = df.sort_values(['driver_id', 'date']).reset_index(drop=True)
df['driver_career_avg'] = (
    df.groupby('driver_id')['finish_position']
    .transform(lambda x: x.shift(1).expanding().mean())
)
df['driver_circuit_avg'] = (
    df.groupby(['driver_id', 'circuit_id'])['finish_position']
    .transform(lambda x: x.shift(1).expanding().mean())
)
df['track_delta'] = df['driver_career_avg'] - df['driver_circuit_avg']

In [18]:
## Constructor Career Average
#Sort key back to `['constructor_id', 'date', 'driver_id']`.

In [19]:
df = df.sort_values(['constructor_id', 'date', 'driver_id']).reset_index(drop=True)
df['constructor_career_avg'] = (
    df.groupby('constructor_id')['finish_position']
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [20]:
## Delta Features & Circuit History Flag

In [21]:
df['driver_vs_team_delta'] = df['driver_career_avg'] - df['constructor_career_avg']
df['driver_form_delta'] = df['driver_rolling_avg_finish'] - df['driver_career_avg']
df['has_circuit_history'] = df['driver_circuit_avg'].notna().astype(int)

In [22]:
## Null Check on Engineered Features

In [23]:
feature_check = [
    'grid_position', 'quali_gap_to_pole', 'made_q2', 'made_q3',
    'driver_rolling_avg_finish', 'constructor_rolling_avg_finish',
    'driver_career_avg', 'driver_circuit_avg', 'track_delta',
    'constructor_career_avg', 'driver_vs_team_delta', 'driver_form_delta',
    'has_circuit_history', 'air_temp', 'track_temp', 'rainfall', 'humidity'
]
df[feature_check].isnull().sum()

grid_position                       2
quali_gap_to_pole                   0
made_q2                             0
made_q3                             0
driver_rolling_avg_finish          31
constructor_rolling_avg_finish     12
driver_career_avg                  31
driver_circuit_avg                730
track_delta                       730
constructor_career_avg             12
driver_vs_team_delta               33
driver_form_delta                  31
has_circuit_history                 0
air_temp                            0
track_temp                          0
rainfall                            0
humidity                            0
dtype: int64

In [24]:
## Fill Remaining Nulls

In [25]:
overall_mean_finish = df['finish_position'].mean()
df['grid_position'] = df['grid_position'].fillna(20)
df['driver_rolling_avg_finish'] = df['driver_rolling_avg_finish'].fillna(overall_mean_finish)
df['driver_career_avg'] = df['driver_career_avg'].fillna(overall_mean_finish)
df['constructor_rolling_avg_finish'] = df['constructor_rolling_avg_finish'].fillna(overall_mean_finish)
df['constructor_career_avg'] = df['constructor_career_avg'].fillna(overall_mean_finish)
df['driver_circuit_avg'] = df['driver_circuit_avg'].fillna(df['driver_career_avg'])
df['track_delta'] = df['track_delta'].fillna(0)
df['driver_form_delta'] = df['driver_form_delta'].fillna(0)
df['driver_vs_team_delta'] = df['driver_vs_team_delta'].fillna(0)
print(df[feature_check].isnull().sum().sum())

0


In [26]:
## Build Modeling Dataframe
#Kept `race_id`, `driver_id`, `finish_position`, `status` alongside features — needed for relevance labels, group construction, and identifying predictions afterward. `points_finish` kept too in case we want a quick v1-vs-v2 sanity comparison later.

In [27]:
feature_columns = [
    'grid_position', 'quali_gap_to_pole', 'made_q2', 'made_q3',
    'driver_rolling_avg_finish', 'constructor_rolling_avg_finish',
    'driver_career_avg', 'driver_circuit_avg', 'track_delta',
    'constructor_career_avg', 'driver_vs_team_delta', 'driver_form_delta',
    'has_circuit_history',
    'air_temp', 'track_temp', 'rainfall', 'humidity',
    'constructor_id', 'circuit_id'
]
id_columns = ['race_id', 'driver_id', 'finish_position', 'status', 'points_finish', 'season', 'round', 'date']

model_df = df[feature_columns + id_columns].copy()
print(model_df.isnull().sum().sum())
print(model_df.shape)

2
(1838, 27)


In [28]:
## Feature Correlation Check (vs v1 target, for reference)

In [29]:
numeric_features = [
    'grid_position', 'quali_gap_to_pole', 'made_q2', 'made_q3',
    'driver_rolling_avg_finish', 'constructor_rolling_avg_finish',
    'driver_career_avg', 'driver_circuit_avg', 'track_delta',
    'constructor_career_avg', 'driver_vs_team_delta', 'driver_form_delta',
    'has_circuit_history', 'air_temp', 'track_temp', 'rainfall', 'humidity'
]
correlations = model_df[numeric_features + ['points_finish']].corr()['points_finish'].drop('points_finish').sort_values(ascending=False)
print(correlations)

made_q3                           0.566021
made_q2                           0.403039
has_circuit_history               0.098731
track_delta                       0.044798
track_temp                        0.000983
air_temp                          0.000243
humidity                         -0.000269
rainfall                         -0.000610
driver_vs_team_delta             -0.078785
driver_form_delta                -0.138742
quali_gap_to_pole                -0.281091
driver_circuit_avg               -0.441858
constructor_rolling_avg_finish   -0.490240
constructor_career_avg           -0.505975
driver_rolling_avg_finish        -0.514658
driver_career_avg                -0.531572
grid_position                    -0.557053
Name: points_finish, dtype: float64


In [30]:
#since 2 nulls survived lets see them

In [31]:
df[feature_check].isnull().sum()[df[feature_check].isnull().sum() > 0]

Series([], dtype: int64)

In [32]:
model_df.isnull().sum()[model_df.isnull().sum() > 0]

finish_position    2
dtype: int64

In [33]:
df[df['status'] != 'Finished'][['driver_id', 'race_id', 'finish_position', 'status']].head(15)

,driver_id,race_id,finish_position,status
2,bottas,73,15.0,Cooling system
7,zhou,75,15.0,+1 Lap
9,zhou,76,20.0,Water leak
11,zhou,77,19.0,Power loss
13,zhou,78,16.0,+1 Lap
14,bottas,79,11.0,+1 Lap
15,zhou,79,18.0,Hydraulics
18,bottas,81,17.0,Gearbox
19,zhou,81,19.0,Collision
20,bottas,82,11.0,+1 Lap


In [34]:
df[df['finish_position'].isnull()][['driver_id', 'race_id', 'season', 'round', 'status']]

,driver_id,race_id,season,round,status
433,stroll,17,2023,15,Withdrew
730,mick_schumacher,73,2022,2,Withdrew


In [35]:
# both are Withdrew, which is a distinct case from a normal DNF: the driver didn't finish, but they also weren't running and classified somewhere in the order before retiring with a mechanical issue like the Bottas/Zhou rows. A withdrawal typically means they pulled out before completing enough laps to be classified at all, which is why there's no position recorded — there genuinely isn't one to record, this isn't missing data, it's data that doesn't exist.

In [36]:
field_size = df.groupby('race_id')['driver_id'].transform('count')
df['finish_position_filled'] = df['finish_position'].fillna(field_size + 1)

In [37]:
## Handle Withdrawn Drivers (no recorded finish_position)
#`Withdrew` status means no classification was given — not missing data, genuinely doesn't exist. Assigned one worse than the last classified finisher in that race, using actual field size per race rather than assuming 20.

In [38]:
field_size = df.groupby('race_id')['driver_id'].transform('count')
df['finish_position_filled'] = df['finish_position'].fillna(field_size + 1)

# sanity check the two rows we found
df[df['finish_position'].isnull()][['driver_id', 'race_id', 'finish_position_filled']]

,driver_id,race_id,finish_position_filled
433,stroll,17,21.0
730,mick_schumacher,73,21.0


In [39]:
## Build Relevance Label
#Higher relevance = better finish. Using `finish_position_filled` so withdrawals get a real (worst-in-race) value instead of breaking the formula with NaN.

In [40]:
df['relevance'] = 21 - df['finish_position_filled']

# sanity check: P1 should get relevance 20, last place / withdrawn should be lowest
df[['driver_id', 'race_id', 'finish_position_filled', 'relevance']].sort_values('relevance').head(10)

,driver_id,race_id,finish_position_filled,relevance
433,stroll,17,21.0,0.0
730,mick_schumacher,73,21.0,0.0
1121,hamilton,85,20.0,1.0
995,norris,23,20.0,1.0
33,zhou,88,20.0,1.0
1080,piastri,63,20.0,1.0
24,bottas,84,20.0,1.0
1811,sainz,57,20.0,1.0
1019,norris,34,20.0,1.0
9,zhou,76,20.0,1.0


In [41]:
print((df.index == model_df.index).all())

True


In [42]:
## Bring Relevance into model_df, Sort Chronologically
#Index alignment confirmed above, so direct column assignment is safe. Sorting by date then race_id afterward guarantees rows from the same race sit contiguously, with races in chronological order — required for both the group array and a clean date-based train/test split.

In [43]:
model_df['relevance'] = df['relevance']
model_df['finish_position_filled'] = df['finish_position_filled']

model_df = model_df.sort_values(['date', 'race_id']).reset_index(drop=True)

model_df[['race_id', 'driver_id', 'date', 'finish_position_filled', 'relevance']].head(25)

,race_id,driver_id,date,finish_position_filled,relevance
0,72,bottas,2022-03-20,6.0,15.0
1,72,zhou,2022-03-20,10.0,11.0
2,72,gasly,2022-03-20,20.0,1.0
3,72,tsunoda,2022-03-20,8.0,13.0
4,72,alonso,2022-03-20,9.0,12.0
5,72,ocon,2022-03-20,7.0,14.0
6,72,hulkenberg,2022-03-20,17.0,4.0
7,72,stroll,2022-03-20,12.0,9.0
8,72,leclerc,2022-03-20,1.0,20.0
9,72,sainz,2022-03-20,2.0,19.0


In [44]:
## Train/Test Split (Date-Based, Same as v1)
#2022-2024 train, 2025 test — consistent with v1, and required here too since this is time-series data.

In [45]:
train_df = model_df[model_df['season'] < 2025].reset_index(drop=True)
test_df = model_df[model_df['season'] == 2025].reset_index(drop=True)

print(train_df.shape, test_df.shape)

(1359, 29) (479, 29)


In [46]:
## Build Group Arrays
#XGBoost's ranking objective needs an array of group sizes — how many consecutive rows belong to each race — computed separately per split since groups can't span the train/test boundary. Using `groupby(...).size()` on `race_id`, which works correctly because rows are already sorted contiguously by race within each split.

In [47]:
train_groups = train_df.groupby('race_id').size().values
test_groups = test_df.groupby('race_id').size().values

print(train_groups.sum(), len(train_df))  # should match
print(test_groups.sum(), len(test_df))    # should match
print(train_groups[:5], test_groups[:5])

1359 1359
479 479
[20 20 20 20 20] [20 20 20 20 20]


In [48]:
# confirm race_ids appear in increasing order as rows progress (i.e. groupby('race_id').size() preserves row order)
print((train_df['race_id'].values == train_df.sort_values('race_id')['race_id'].values).all())
print((test_df['race_id'].values == test_df.sort_values('race_id')['race_id'].values).all())

False
True


In [49]:
# find where race_id order disagrees with row order in train_df
check = train_df[['race_id', 'date', 'season', 'round']].copy()
check['prev_race_id'] = check['race_id'].shift(1)
mismatches = check[check['race_id'] < check['prev_race_id']]
mismatches

,race_id,date,season,round,prev_race_id
440,2,2023-03-05,2023,1,93.0
880,1,2024-03-02,2024,1,24.0


In [50]:
# verify the suspicion: does race_id repeat across seasons?
df.groupby('race_id')['season'].nunique().sort_values(ascending=False).head()

race_id
1    1
2    1
3    1
4    1
5    1
Name: season, dtype: int64

In [51]:
df[['race_id', 'season', 'round', 'date']].drop_duplicates().sort_values('race_id').head(20)

,race_id,season,round,date
264,1,2024,1,2024-03-02
44,2,2023,1,2023-03-05
312,3,2025,1,2025-03-16
46,4,2023,2,2023-03-19
48,5,2023,3,2023-04-02
50,6,2023,4,2023-04-30
52,7,2023,5,2023-05-07
54,8,2023,6,2023-05-28
56,9,2023,7,2023-06-04
58,10,2023,8,2023-06-18


In [52]:
#fix 

train_groups = train_df.groupby('race_id', sort=False).size().values
test_groups = test_df.groupby('race_id', sort=False).size().values

print(train_groups.sum(), len(train_df))
print(test_groups.sum(), len(test_df))

1359 1359
479 479


In [53]:
#verify
# rebuild the monotonicity-style check, but against row position instead of race_id value
train_df['_row_race_change'] = (train_df['race_id'] != train_df['race_id'].shift(1)).cumsum()
expected_groups = train_df.groupby('_row_race_change', sort=False).size().values

print((train_groups == expected_groups).all())
print(len(train_groups), len(expected_groups))

True
68 68


In [54]:
train_df = train_df.drop(columns=['_row_race_change'])

In [55]:
## Build Feature Matrices
#Same feature_columns as before. Categoricals (constructor_id, circuit_id) need one-hot encoding, same approach as v1 — combine train+test first so column sets match.

In [56]:
categorical_cols = ['constructor_id', 'circuit_id']
numeric_cols = [c for c in feature_columns if c not in categorical_cols]

combined = pd.concat([train_df[feature_columns], test_df[feature_columns]], axis=0)
combined_encoded = pd.get_dummies(combined, columns=categorical_cols)

X_train = combined_encoded.iloc[:len(train_df)].reset_index(drop=True)
X_test = combined_encoded.iloc[len(train_df):].reset_index(drop=True)

print(X_train.shape, X_test.shape)

(1359, 55) (479, 55)


In [57]:
## Build DMatrix Objects with Groups

In [58]:
import xgboost as xgb

dtrain = xgb.DMatrix(X_train, label=train_df['relevance'])
dtrain.set_group(train_groups)

dtest = xgb.DMatrix(X_test, label=test_df['relevance'])
dtest.set_group(test_groups)

print(dtrain.num_row(), dtrain.get_label().shape)
print(dtest.num_row(), dtest.get_label().shape)

1359 (1359,)
479 (479,)


In [59]:
params = {
    'objective': 'rank:ndcg',
    'eval_metric': 'ndcg@10',
    'eta': 0.1,
    'max_depth': 4,
    'min_child_weight': 5,
}

In [60]:
evals_result = {}
bst = xgb.train(
    params,
    dtrain,
    num_boost_round=200,
    evals=[(dtrain, 'train'), (dtest, 'test')],
    evals_result=evals_result,
    verbose_eval=20
)

[0]	train-ndcg@10:0.70202	test-ndcg@10:0.66492
[20]	train-ndcg@10:0.93643	test-ndcg@10:0.84317
[40]	train-ndcg@10:0.94570	test-ndcg@10:0.85545
[60]	train-ndcg@10:0.94815	test-ndcg@10:0.84201
[80]	train-ndcg@10:0.95219	test-ndcg@10:0.83757
[100]	train-ndcg@10:0.95640	test-ndcg@10:0.84272
[120]	train-ndcg@10:0.96227	test-ndcg@10:0.85440
[140]	train-ndcg@10:0.96371	test-ndcg@10:0.83764
[160]	train-ndcg@10:0.96484	test-ndcg@10:0.83143
[180]	train-ndcg@10:0.96507	test-ndcg@10:0.84493
[199]	train-ndcg@10:0.96555	test-ndcg@10:0.84410


In [61]:
evals_result = {}
bst = xgb.train(
    params,
    dtrain,
    num_boost_round=200,
    evals=[(dtrain, 'train'), (dtest, 'test')],
    evals_result=evals_result,
    early_stopping_rounds=20,
    verbose_eval=20
)

print("Best iteration:", bst.best_iteration)
print("Best test NDCG@10:", bst.best_score)

[0]	train-ndcg@10:0.70202	test-ndcg@10:0.66492
[20]	train-ndcg@10:0.93643	test-ndcg@10:0.84317
[26]	train-ndcg@10:0.93970	test-ndcg@10:0.84526
Best iteration: 6
Best test NDCG@10: 0.855727554882384


In [62]:
## Feature Importance

In [63]:
importance = bst.get_score(importance_type='gain')
importance_df = pd.DataFrame(importance.items(), columns=['feature', 'gain']).sort_values('gain', ascending=False)
importance_df.head(20)

,feature,gain
1,quali_gap_to_pole,19.043955
7,constructor_career_avg,5.462823
2,driver_rolling_avg_finish,3.837285
0,grid_position,3.284277
16,constructor_id_red_bull,2.031348
4,driver_career_avg,1.906014
8,driver_vs_team_delta,1.653585
3,constructor_rolling_avg_finish,1.590288
13,humidity,1.463181
9,driver_form_delta,1.403532


In [64]:
## Inspect One Race: Predicted vs Actual Order

In [65]:
preds = bst.predict(dtest)
test_df['predicted_score'] = preds

# pick a race to inspect — let's grab the first race in the test set (2025 round 1)
sample_race_id = test_df['race_id'].iloc[0]

race_view = test_df[test_df['race_id'] == sample_race_id][
    ['driver_id', 'finish_position_filled', 'relevance', 'predicted_score']
].copy()

race_view['predicted_rank'] = race_view['predicted_score'].rank(ascending=False).astype(int)
race_view = race_view.sort_values('predicted_rank')

race_view

,driver_id,finish_position_filled,relevance,predicted_score,predicted_rank
8,norris,1.0,20.0,0.847602,1
15,max_verstappen,2.0,19.0,0.505859,2
9,piastri,9.0,12.0,0.492761,3
11,russell,3.0,18.0,0.206867,4
5,leclerc,8.0,13.0,-0.224719,5
4,hamilton,10.0,11.0,-0.703008,6
19,sainz,18.0,3.0,-0.707037,7
10,antonelli,4.0,17.0,-0.817813,8
13,tsunoda,12.0,9.0,-0.990799,9
18,albon,5.0,16.0,-1.165340,10
